# Object Following - Live Demo

In this notebook we'll show how you can follow an object with JetBot!  We'll use a pre-trained neural network
that was trained on the [COCO dataset](http://cocodataset.org) to detect 90 different common objects.  These include

* Person (index 0)
* Cup (index 47)

and many others (you can check [this file](https://github.com/tensorflow/models/blob/master/research/object_detection/data/mscoco_complete_label_map.pbtxt) for a full list of class indices).  The model is sourced from the [TensorFlow object detection API](https://github.com/tensorflow/models/tree/master/research/object_detection),
which provides utilities for training object detectors for custom tasks also!  Once the model is trained, we optimize it using NVIDIA TensorRT on the Jetson Nano.

This makes the network very fast, capable of real-time execution on Jetson Nano!  We won't run through all of the training and optimization steps in this notebook though.

Anyways, let's get started.  First, we'll want to import the ``ObjectDetector`` class which takes our pre-trained SSD engine.

### Compute detections on single camera image

In [ ]:
from jetbot import ObjectDetector

model = ObjectDetector('ssd_mobilenet_v2_coco.engine')

Internally, the ``ObjectDetector`` class uses the TensorRT Python API to execute the engine that we provide.  It also takes care of preprocessing the input to the neural network, as
well as parsing the detected objects.  Right now it will only work for engines created using the ``jetbot.ssd_tensorrt`` package. That package has the utilities for converting
the model from the TensorFlow object detection API to an optimized TensorRT engine.

Next, let's initialize our camera.  Our detector takes 300x300 pixel input, so we'll set this when creating the camera.

> Internally, the Camera class uses GStreamer to take advantage of Jetson Nano's Image Signal Processor (ISP).  This is super fast and offloads
> a lot of the resizing computation from the CPU. 

In [ ]:
from jetbot import Camera

camera = Camera.instance(width=300, height=300)

Now, let's execute our network using some camera input.  By default the ``ObjectDetector`` class expects ``bgr8`` format that the camera produces.  However,
you could override the default pre-processing function if your input is in a different format.

In [ ]:
detections = model(camera.value)

print(detections)

If there are any COCO objects in the camera's field of view, they should now be stored in the ``detections`` variable.

### Display detections in text area

We'll use the code below to print out the detected objects.

In [ ]:
from IPython.display import display
import ipywidgets.widgets as widgets

detections_widget = widgets.Textarea()

detections_widget.value = str(detections)

display(detections_widget)

You should see the label, confidence, and bounding box of each object detected in each image.  There's only one image (our camera) in this example. 


To print just the first object detected in the first image, we could call the following

> This may throw an error if no objects are detected

In [ ]:
image_number = 0
object_number = 0

print(detections[image_number][object_number])

### Control robot to follow central object(已加裝鎖定狀態機)

本節已相對官方版本改寫。原本的流程是:

1. 偵測符合類別的物件
2. 每幀獨立選「離畫面中心最近」的框當目標
3. 有目標就轉向跟隨,**沒目標就盲目前進**
4. 被擋住就左轉

改成:

1. 偵測符合類別的物件
2. **跨幀連續性追蹤**:比對候選框與上一幀鎖定框的位置/大小,延續同一個目標(項目1)
3. **鎖定狀態機** `waiting → confirming → locked`:目標需水平置中 ±15%、框高超過距離門檻,
   並連續維持 0.75 秒才算鎖定(項目2)
4. 鎖定進度以 **LED 閃爍頻率**呈現,鎖定完成時**蜂鳴器短鳴一聲**;
   蜂鳴器只在這一刻使用(項目3)
5. 跟丟但畫面還有其他人 → 退而選最近候選框重新鎖定,**不發任何提示音**(項目4)
6. **畫面中完全沒人 → 原地停住(不再盲目前進)+ 通知照護者**(項目5)
7. 碰撞避障維持官方的 early-return 優先權覆蓋架構,但避障期間會凍結跟丟計時,
   結束後直接銜接回原本鎖定的目標,不用重跑鎖定流程(項目6)
8. 判定繞不過去時 → 進入 `impassable_wait`,停止空轉 + 通知照護者,
   **同樣不對目標者發警示音**(項目7)。碰撞模型只需要分「擋住/沒擋住」,
   不必分辨樓梯——JetBot 對樓梯和牆角一樣過不去,所以改用時間特徵判定,
   不需要額外的感測器。兩個條件任一成立即判定:
   **(a)** 連續被擋超過 5 秒;**(b)** 8 秒滑動視窗內被擋比例超過一半(擺盪偵測);
   **(c)** 連續繞行 3 次仍過不去
9. **繞行**:轉開之後強制直行一段(`detour` 狀態的橫移階段),讓車體真的產生橫向
   位移才交還給跟隨邏輯。官方的 `robot.left()` 是原地自轉,車體位置完全沒動,
   若轉開的下一幀就讓跟隨邏輯把車頭拉回目標,會直接正對回障礙物形成擺盪。
   **這個「承諾」才是修掉擺盪的關鍵,不是超音波本身**
10. **VL53L0X 雷射測距**(接在 Arduino 的 I2C 上,不是 Jetson GPIO)。與相機碰撞模型
    以 OR 融合成單一「被擋住」訊號,貢獻是相機給不了的**絕對距離數值**。
    感測器沒接也完全可以運作——上述所有邏輯都不依賴它,距離會是 `None`,
    融合邏輯單獨採信相機判斷。取代原本規劃的 HC-SR04:不用分壓電阻、
    不怕布料吸音、不用在 Python 裡掐秒錶
11. **身分驗證改用螢光雙色標記**(`marker.py`),貼在被跟隨者兩腿的小腿外側。
    取代原本延到 v2 的 re-ID。HSV 門檻約 1ms、零訓練資料;YOLO 要 30~50ms,
    而且 JetPack 4.5 的 Python 3.6 裝不了 ultralytics
12. **回饋改用交通號誌模組**(紅/黃/綠)。順帶解決一個缺陷:單顆 LED 時
    `waiting` 與 `impassable_wait` 都是慢閃、分不出來,但意義天差地遠

決策邏輯放在 `follow_state.py`(純 Python,可在筆電上跑 `test_follow_state.py` 離線驗證),
硬體輸出放在 `feedback.py`。這個 callback 只負責「取影像 → 問狀態機 → 執行動作」。

跌倒偵測(IMU)依需求文件第 111 行,不在本 notebook 內,另開獨立模組。

先載入碰撞避障模型(與官方相同,未更動)。

In [ ]:
import json
import os

import torch
import torchvision
import torch.nn.functional as F
import cv2
import numpy as np

# --- 避障模型 --------------------------------------------------------------
#
# ⚠️⚠️ 目前指向的是「77 張訓練的**煙霧測試**模型」。
#      它只能用來確認管線通不通、以及**量 FPS**,避障判斷不可信:
#        - 測試集只有 15 張,而且全是同一個場景
#        - meta 記錄的類別數是 blocked:30 / free:47,但那 4 張標籤修正是在
#          訓練**之後**才做的,現在資料集是 26/51 —— 標籤已經過期
#      **不要當成果、不要寫進報告。**
#
#      避障資料補到 240 張、重新訓練之後,只要改下面 COLLISION_MODEL_PATH
#      這一個字串,其他都不用動。
#
# FPS 不受權重好壞影響 —— 同樣的架構、同樣的張量形狀、同樣的運算量,
# 所以現在就可以量,不必等資料集補完。
COLLISION_ARCH = 'resnet18'      # 已定案:不是官方的 alexnet
COLLISION_MODEL_PATH = '../collision_avoidance/best_model_resnet18.pth'

# 為什麼是 ResNet18 不是官方的 AlexNet:
#   參數 1100 萬 vs 6100 萬、推論快 2~3 倍、ImageNet 準確率反而高約 13 個百分點。
#   更小、更快、更準,沒有取捨。
#
# 注意用 pretrained=False 而不是新版的 weights=None:
#   JetPack 4.5 上的 torchvision 版本較舊,沒有 weights= 這個參數。
#   (train_collision.py 在筆電上跑,那邊用的是新 API,兩邊不一樣是正常的)
if COLLISION_ARCH == 'resnet18':
    collision_model = torchvision.models.resnet18(pretrained=False)
    collision_model.fc = torch.nn.Linear(collision_model.fc.in_features, 2)
else:
    collision_model = torchvision.models.alexnet(pretrained=False)
    collision_model.classifier[6] = torch.nn.Linear(
        collision_model.classifier[6].in_features, 2)

collision_model.load_state_dict(torch.load(COLLISION_MODEL_PATH))

device = torch.device('cuda')

# ⚠️ .eval() 不能省。官方 notebook 漏了這一行,對 AlexNet 只是多一點隨機性
#    (Dropout),但對 ResNet18 會**直接出錯**:
#      訓練模式下的 BatchNorm 用「當下這一批」的統計量做正規化,
#      而我們每次只餵一張(batch=1),等於拿單一樣本的統計量去正規化它自己。
#      結果與訓練時完全不同,而且會順便污染 running statistics。
#    沒有錯誤訊息,只會讓避障判斷變成亂數。
collision_model = collision_model.to(device).eval()

# --- 與訓練端對照 ----------------------------------------------------------
# train_collision.py 會把架構、類別編號、輸入尺寸、正規化參數存成 meta.json,
# 目的就是讓推論端可以對照。最重要的是 class_to_idx:
# 下面 execute() 用 softmax(...)[0] 取「被擋住」的機率,前提是 blocked=0。
# 資料夾一旦改名,編號會反過來而且**不會報錯** —— 機器人會變成看到路就停、
# 看到牆就衝。所以在這裡驗一次。
INPUT_W, INPUT_H = 224, 168      # (寬, 高)

_meta_path = os.path.splitext(COLLISION_MODEL_PATH)[0] + '_meta.json'
if os.path.exists(_meta_path):
    with open(_meta_path, encoding='utf-8') as _f:
        _meta = json.load(_f)
    print('避障模型 meta:', _meta['arch'], _meta['class_to_idx'],
          '輸入 %dx%d' % (_meta['input_width'], _meta['input_height']))
    assert _meta['class_to_idx'] == {'blocked': 0, 'free': 1}, \
        '類別編號與推論端假設不符!softmax[0] 取的就不是 blocked 了'
    assert (_meta['input_width'], _meta['input_height']) == (INPUT_W, INPUT_H), \
        '輸入尺寸與訓練時不符,模型看到的畫面形狀會跟訓練時不同'
    if _meta['arch'] != COLLISION_ARCH:
        print('[警告] meta 的架構是 %s,但這裡建的是 %s'
              % (_meta['arch'], COLLISION_ARCH))
else:
    print('[警告] 找不到 %s,無法驗證類別編號與輸入尺寸' % _meta_path)

mean = 255.0 * np.array([0.485, 0.456, 0.406])
stdev = 255.0 * np.array([0.229, 0.224, 0.225])

normalize = torchvision.transforms.Normalize(mean, stdev)


def preprocess(camera_value):
    global device, normalize
    x = camera_value
    # cv2.resize 吃 (寬, 高);train_collision.py 的 transforms.Resize 吃 (高, 寬)。
    # 順序相反,很容易寫錯。三處尺寸必須一致:
    #   拍照 Camera.instance(width=224, height=168)
    #   訓練 Resize((168, 224))
    #   推論 cv2.resize(x, (224, 168))   ← 這裡
    x = cv2.resize(x, (INPUT_W, INPUT_H))
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1))
    x = torch.from_numpy(x).float()
    x = normalize(x)
    x = x.to(device)
    x = x[None, ...]
    return x

Great, now let's initialize our robot so we can control the motors.

In [ ]:
from jetbot import Robot

robot = Robot()

Finally, let's display all the control widgets and connect the network execution function to the camera updates.

### 載入狀態機與回饋硬體

`follow_state.py` / `feedback.py` / `rangefinder.py` / `marker.py` 需與本 notebook
放在同一個資料夾。**修改過這些 `.py` 之後要重啟 kernel 才會生效**（Jupyter 不會重載模組）。

`FollowConfig` 裡有兩類參數：已定案的（置中 ±15%、鎖定 0.75 秒）以及
**待現場實測**的（`min_bbox_height` 距離門檻、追蹤幾何門檻）。實測時直接改這一格的值。

---

#### 2026-09-18 硬體改版

| 原本 | 現在 |
|---|---|
| 單顆 LED（D10） | **交通號誌模組**，紅 D10 / 黃 D9 / 綠 D8 |
| HC-SR04 超音波接 Jetson GPIO | **VL53L0X 雷射 ToF 接 Arduino I2C**（A4/A5） |
| 序列埠 9600、問一句答一句 | **115200、Arduino 以 10Hz 主動推播距離** |
| re-ID 延到 v2 | **螢光雙色標記**（`marker.py`），貼兩腿小腿外側 |

**執行前必做兩件事：**

1. 燒錄新版 `arduino_feedback/arduino_feedback.ino`（鮑率不對會全是亂碼）
2. 跑過 `marker_calibration/` 的校正流程，把實測 HSV 門檻填回 `marker.py`

#### 狀態 → 燈號對照

| 狀態 | 燈 | 給目標者的意思 |
|---|---|---|
| `waiting` | 🔴 慢閃 | 還沒開始跟 |
| `confirming` | 🟡 快閃 | 倒數中，站好別動 |
| `locked` | 🟢 恆亮 | 正常跟隨 |
| `detour` | 🟢 恆亮 ＋ 🟡 慢閃 | 還跟著你，正在繞路 |
| `impassable_wait` | 🔴 恆亮 | 停止，已通知照護者 |

`detour` 的綠燈刻意與 `locked` 相同 —— 繞行期間鎖定並沒有解除，
「還跟著你」的訊息不該改變；黃燈是額外疊上去的資訊。

三色之後才解決的缺陷：單顆 LED 時 `waiting` 與 `impassable_wait` 都是慢閃、
長得一模一樣，但這兩個狀態意義天差地遠。

In [ ]:
import time

import follow_state as fs
from follow_state import FollowStateMachine, FollowConfig
import feedback
import rangefinder

config = FollowConfig()

# --- 已定案(需求文件規格)---
config.center_tolerance = 0.15     # 水平置中容許誤差:畫面寬 ±15%
config.confirm_seconds = 0.75      # 鎖定所需連續時間,規格 0.5~1.0 秒

# --- 待現場實測定案 ---
# TODO: 攝影機安裝角度確定後,實拍 0.5m / 1m / 1.5m 站立距離量測框高比例再定案。
#       下方 execute() 會把目標框的 h= 比例畫在畫面上,量測時直接讀。
config.min_bbox_height = 0.30
config.track_center_max_dist = 0.25
config.track_height_ratio_max = 1.6
config.lost_grace_seconds = 2.0

# 跟隨距離控制:官方版前進速度固定,人停下來時機器人會一路逼近到碰撞模型
# 判定被擋住為止。改成依距離縮放速度,接近就減速、太近就停(但仍會轉向)。
# 距離優先用雷射測距,沒有就退回框高。
# TODO: 兩組門檻都要現場實測。
config.follow_distance_m = 0.80     # 雷射:近於此開始減速
config.stop_distance_m = 0.45       # 雷射:近於此完全停止
config.slow_bbox_height = 0.55      # 框高:大於此開始減速
config.stop_bbox_height = 0.75      # 框高:大於此完全停止
config.distance_trust_offset = 0.15  # 目標偏離中心超過此值就不採信雷射讀數

# 跟丟後的方向性搜尋:直角牆角時目標會先飄到畫面邊緣才被牆擋住,
# 這時朝最後已知的方向原地轉一小段,比立刻停住更有機會找回來。
# 刻意只旋轉不前進——看不到人還往前開是官方版最危險的行為。
config.search_turn_seconds = 1.5   # TODO: 太短轉不到,太長會轉過頭
config.search_min_offset = 0.12    # 消失前偏移小於這個量就不轉(方向不可靠)
SEARCH_TURN_SPEED = 0.25           # 比避障的 0.3 慢,避免轉過頭

# 項目7:判定「繞不過去」的兩個條件,任一成立就停下來通知照護者。
# TODO: 現場實測校正。走廊轉角可能要 3~4 秒才轉得開,設太短會把轉角誤判成死路。
config.impassable_after_blocked_seconds = 5.0   # (a) 連續被擋這麼久
config.block_window_seconds = 8.0               # (b) 擺盪偵測的滑動視窗
config.block_ratio_threshold = 0.5              # (b) 視窗內被擋比例超過此值

# (b) 為什麼必要:robot.left() 是原地自轉,車體沒有橫向位移。障礙物擋在機器人與
# 目標之間時會出現「左轉到不擋住 → 跟隨邏輯把車頭拉回目標 → 又正對障礙物」的
# 來回擺盪,每次被擋都很短,(a) 永遠不會觸發,機器人會無限卡住且不通知任何人。

# 繞行:轉開之後強制直行一段,讓車體真的產生橫向位移,這段期間不理會目標的拉力。
# 這才是修掉擺盪的關鍵——不是測距感測器本身。
config.detour_advance_seconds = 1.5   # TODO: 太短繞不過去,太長會偏離目標太遠
config.max_detour_attempts = 3        # 連續繞這麼多次還過不去就放棄並通知
config.detour_reset_seconds = 10.0

# --- 硬體:紅綠燈 / 蜂鳴器 / 雷射測距(全部接在 Arduino 上)---
#
# 2026-09-18 改版。序列埠 115200,由 feedback.ArduinoBackend 獨佔並開背景執行緒
# 讀取;VL53L0X 的距離由 Arduino 以 10Hz 主動推播,camera callback 只讀快取值,
# 完全不阻塞。
#
# ⚠️ 韌體要先燒錄新版 arduino_feedback.ino,鮑率才對得上,否則會全是亂碼。
fb = feedback.Feedback()
print('回饋硬體後端:', fb.backend.name)

sm = FollowStateMachine(config)

# --- VL53L0X 雷射測距 ---
#
# 已經不需要舊版 ENABLE_SONAR 那種安全開關了:HC-SR04 時代的風險是
# 「未接線的 GPIO 浮接會讀到隨機電位、回報假的近距離」,而 I2C 沒接就是沒回應,
# 韌體會回報 vl53l0x=absent,distance 永遠是 None,融合邏輯自動單獨採信相機。
#
# 這個開關留著只是為了除錯時能手動切掉測距、隔離問題來源。
ENABLE_RANGEFINDER = True
RANGE_BLOCK_DISTANCE = 0.35   # 公尺。TODO: 依車體長度與煞停距離實測

ultrasonic = rangefinder.RangeFinder(fb) if ENABLE_RANGEFINDER else None
print('雷射測距:', '啟用' if ENABLE_RANGEFINDER else '手動關閉(僅用相機碰撞模型)')

# --- 身分驗證:螢光雙色標記(取代原本規劃的 re-ID)---
#
# 被跟隨者**兩腿的小腿外側各貼一張**「上桃紅 / 下螢光黃」的標記,用 HSV 門檻找。
# 不用 YOLO 的理由見 marker.py 開頭:這是顏色查表問題、不是辨識問題,
# HSV 約 1ms 且零訓練資料;YOLO 要 30~50ms,而且 JetPack 4.5 的 Python 3.6
# 裝不了 ultralytics。兩個神經網路已經吃滿 FPS 預算,不能再加第三個。
#
# signature_fn 維持預設不變:標記是絕對身分,不需要抽特徵去比對。
#
# ⚠️ marker.py 裡的 HSV 門檻目前是**估計的起點,不是實測值**。
#    上車前先跑 marker_calibration/ 的校正流程,把實測數字填回去。
#    沒校正就上車,最可能的症狀是「完全鎖不上」或「沒貼標記的人也鎖得上」。
# 🔻 第一次上車先設 False。門檻還是估計值,開著很可能完全鎖不上,
#    那就測不到跟隨行為了(FPS 照樣量得到,但框高門檻校正不了)。
#    關掉之後 identity_score 回到預設值(永遠回 1.0),行為與加標記之前完全相同。
#    跑完 marker_calibration/ 的校正、把實測門檻填回 marker.py 之後,改回 True。
ENABLE_MARKER = False

verifier = None
if ENABLE_MARKER:
    import marker
    verifier = marker.MarkerVerifier()
    sm.identity_score = verifier
    config.identity_threshold = 0.5

print('identity_threshold =', config.identity_threshold,
      '(標記驗證已啟用)' if ENABLE_MARKER else '(0.0 代表不做身分驗證)')

### 車外通報：MQTT

已定案由**車外電腦**負責判斷異常並推播 LINE，車上只負責**送出原始狀態**。
傳輸走 MQTT，組員提供 `sender.py`，我們這邊只呼叫 `sender.send(payload)`。

**車上不再直接推 LINE。** 否則車上與車外會各推一次，照護者收到重複訊息。

#### 送出的資料格式（與組員的約定）

```python
{
  "state":      "locked",              # 五種之一
  "events":     ["lock_acquired"],     # 這一幀發生的事件，可能是空陣列
  "has_target": True,                  # 畫面上有沒有鎖定中的目標
  "readings": {
    "blocked_prob": 0.12,              # 碰撞模型機率
    "distance_m":   0.90,              # 超音波距離，未啟用時為 null
    "speed_scale":  1.0,               # 速度倍率，0 代表因為太近而停住
    "target_h":     0.42               # 目標框高，可當距離代理，無目標時為 null
  }
}
```

`events` 是**陣列**不是字串，因為同一幀可能同時發生多件事。可能的值：
`lock_acquired`、`lock_lost`、`relocked`、`searching`、`enter_confirming`、
`cancel_confirming`、`enter_detour`、`detour_advance`、`detour_done`、
`enter_impassable`、`notify_impassable`、`notify_no_target`、`resume_follow`。

#### 送出時機

**車上已經做好節流**：狀態改變或有事件時立刻送，其餘情況每秒最多一次。
所以 `sender.py` 可以只負責連線與發送，不必再自己節流。

#### 還沒拿到 sender.py 也能跑

下一格會嘗試 `import sender`，失敗就自動退回 console 印字，notebook 照常運作。

In [ ]:
# 車上只送資料，不直接推 LINE。判斷異常與推播都在車外電腦。
try:
    import sender                      # 組員提供：負責 MQTT 連線與發送
    print('sender 已載入，車外通報啟用')
except Exception as _e:
    sender = None
    print('sender 尚未就緒（%s），車外通報退回 console 印字' % _e)

REPORT_INTERVAL = 1.0      # 沒有狀態改變時，最多多久送一次（秒）

_report_state = {'state': None, 'at': 0.0, 'warned': False}


def report_to_caregiver(action, prob_blocked, distance_m, now):
    """送一筆狀態給車外電腦。

    節流規則：狀態改變、或這一幀有事件 → 立刻送；否則每 REPORT_INTERVAL 秒一次。
    這樣既不會漏掉關鍵時刻，也不會用 10Hz 灌爆網路。

    通報失敗絕對不能影響跟隨本身，所以整段包在 try 裡面——
    車子的安全不該取決於 Wi-Fi 通不通。
    """
    changed = (action.state != _report_state['state']) or bool(action.events)
    if not changed and (now - _report_state['at']) < REPORT_INTERVAL:
        return

    target_h = None
    if action.target is not None:
        target_h = round(fs.bbox_height(action.target['bbox']), 3)

    payload = {
        'state': action.state,
        'events': list(action.events),
        'has_target': action.target is not None,
        'readings': {
            'blocked_prob': round(prob_blocked, 3),
            'distance_m': None if distance_m is None else round(distance_m, 3),
            'speed_scale': round(action.speed_scale, 3),
            'target_h': target_h,
        },
    }

    _report_state['state'] = action.state
    _report_state['at'] = now

    if sender is None:
        if action.events:                      # 沒有 sender 時只印關鍵事件，不洗版
            print('[report] %s %s' % (payload['state'], payload['events']))
        return

    try:
        sender.send(payload)
        _report_state['warned'] = False
    except Exception as e:
        if not _report_state['warned']:        # 只警告一次，避免斷線時每秒洗版
            print('[sender] 送出失敗（%s），跟隨不受影響' % e)
            _report_state['warned'] = True

### 主 callback

`label_widget` 維持程式碼原本的預設值 `1`(COCO 的 person)。
notebook 說明文字寫的 "Person (index 0)" 與程式碼不一致,依需求文件第 110 行以程式碼為準,
但部署時務必實測確認框有正確框到人。

畫面上的視覺輔助:

- 藍框:所有偵測到的物件(官方行為)
- 綠框:目前的目標;`locked` 時為粗框,`confirming` 時為細框
- 黃色直線:置中容許區間 ±15% 的邊界,目標框中心落在兩線之間才可能進入 confirming
- 左上角文字:目前狀態、目標框高比例 `h=`(**這就是實測距離門檻要讀的數字**)

In [ ]:
from collections import deque

from jetbot import bgr8_to_jpeg

blocked_widget = widgets.FloatSlider(min=0.0, max=1.0, value=0.0, description='blocked')
image_widget = widgets.Image(format='jpeg', width=300, height=300)
label_widget = widgets.IntText(value=1, description='tracked label')
speed_widget = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, description='speed')
turn_gain_widget = widgets.FloatSlider(value=0.8, min=0.0, max=2.0, description='turn gain')
state_widget = widgets.Text(value='waiting', description='state')
distance_widget = widgets.FloatText(value=-1.0, description='laser (m)')

# 構想書目標 A 要求 >= 15 FPS,而這是整個專案最大的未知數。
# 取最近 30 幀的平均,單幀抖動不會讓數字跳來跳去。
fps_widget = widgets.FloatText(value=0.0, description='FPS', disabled=True)
_frame_times = deque(maxlen=30)

display(widgets.VBox([
    widgets.HBox([image_widget, blocked_widget]),
    widgets.HBox([state_widget, distance_widget, fps_widget]),
    label_widget,
    speed_widget,
    turn_gain_widget
]))

width = int(image_widget.width)
height = int(image_widget.height)


def apply_motor(action):
    """狀態機只決定「要做什麼」,實際馬達指令集中在這裡。"""
    if action.motor == fs.MOTOR_STEER:
        # 前進速度依距離縮放(太近會變成 0),但轉向量**不**縮放——
        # 這樣即使停下來,車頭仍會繼續轉動跟著人,不會因為停住而跟丟。
        base = float(speed_widget.value) * action.speed_scale
        cx = action.steer
        robot.set_motors(base + turn_gain_widget.value * cx,
                         base - turn_gain_widget.value * cx)
    elif action.motor == fs.MOTOR_TURN_LEFT:
        robot.left(SEARCH_TURN_SPEED if action.state == fs.STATE_LOCKED else 0.3)
    elif action.motor == fs.MOTOR_TURN_RIGHT:
        # 跟丟後朝最後已知方向找回目標。只在 locked 狀態會出現,
        # 用比避障慢的速度,避免一口氣轉過頭又把人轉出畫面。
        robot.right(SEARCH_TURN_SPEED)
    elif action.motor == fs.MOTOR_FORWARD:
        # 繞行的橫移階段:直行讓車體真的偏開,期間刻意不理會目標的拉力
        robot.forward(float(speed_widget.value))
    else:
        # 項目5/7:找不到人、或判定繞不過去 → 原地停住,不是官方的 robot.forward()
        robot.stop()


def handle_events(events):
    """把狀態機事件接到實際硬體與通知。

    需求文件第 112-113 行:蜂鳴器只在「鎖定完成」使用。
    EVENT_RELOCKED(跟丟後重新鎖定)刻意不接任何輸出。
    """
    for event in events:
        if event == fs.EVENT_LOCK_ACQUIRED:
            fb.beep_locked()                       # ← 唯一會響蜂鳴器的地方
            feedback.notify_caregiver('已完成鎖定,開始跟隨')
        elif event == fs.EVENT_ENTER_CONFIRMING:
            feedback.notify_caregiver('偵測到目標,鎖定確認中',
                                      key='confirming', cooldown=30.0)
        elif event == fs.EVENT_LOCK_LOST:
            feedback.notify_caregiver('目標已離開偵測範圍,解除鎖定並原地等待')
        elif event == fs.EVENT_NOTIFY_NO_TARGET:
            feedback.notify_caregiver('畫面中找不到人,機器人原地等待中')
        elif event == fs.EVENT_NOTIFY_IMPASSABLE:
            feedback.notify_caregiver('遇到無法跨越的障礙,已暫停跟隨')
        elif event == fs.EVENT_RESUME_FOLLOW:
            feedback.notify_caregiver('障礙解除,恢復跟隨')
        # EVENT_RELOCKED / EVENT_CANCEL_CONFIRMING 不做任何輸出:
        # 不對目標者發警示音,也不必打擾照護者


def draw_overlay(image, action, now):
    """畫置中容許區間與狀態文字,方便現場實測讀數字。"""
    lo = int(width * (0.5 - config.center_tolerance))
    hi = int(width * (0.5 + config.center_tolerance))
    cv2.line(image, (lo, 0), (lo, height), (0, 220, 220), 1)
    cv2.line(image, (hi, 0), (hi, height), (0, 220, 220), 1)

    text = action.state
    if action.target is not None:
        bbox = action.target['bbox']
        thickness = 5 if action.state == fs.STATE_LOCKED else 2
        cv2.rectangle(image,
                      (int(width * bbox[0]), int(height * bbox[1])),
                      (int(width * bbox[2]), int(height * bbox[3])),
                      (0, 255, 0), thickness)
        text += ' h=%.2f' % fs.bbox_height(bbox)
    if action.state == fs.STATE_CONFIRMING:
        text += ' %d%%' % int(100 * sm.confirm_progress(now))
    if action.motor == fs.MOTOR_STEER:
        text += ' v=%.2f' % action.speed_scale

    # 身分標記:洋紅色細框 = 這一幀有抓到標記,文字後面會多一個 M。
    # 現場校正 HSV 門檻時就看這個框有沒有穩定出現——
    # 框一直在閃 = 門檻太嚴;框跑到褲子或地板上 = 門檻太鬆。
    if verifier is not None:
        marker.draw_debug(image, verifier.last_result)
        if verifier.last_result.found:
            text += ' M%d' % verifier.last_result.pink_area

    cv2.putText(image, text, (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)


def track_fps(now):
    """滾動平均。量 FPS 時讀 fps_widget,不要自己拿碼表算。"""
    _frame_times.append(now)
    if len(_frame_times) >= 2:
        span = _frame_times[-1] - _frame_times[0]
        if span > 0:
            fps_widget.value = round((len(_frame_times) - 1) / span, 1)


def execute(change):
    image = change['new']
    now = time.time()

    # --- 碰撞避障:維持官方的 early-return 優先權覆蓋架構(項目6,不改)---
    #
    # no_grad 不能省。少了它,PyTorch 每一幀都會為這個網路建一次 autograd 計算圖,
    # 而我們從頭到尾不做反向傳播,那些記憶體與時間全是白花的。
    # 官方 notebook 只寫 .detach(),那是「建完再丟掉」,不是「不要建」。
    with torch.no_grad():
        collision_output = collision_model(preprocess(image)).cpu()
    prob_blocked = float(F.softmax(collision_output.flatten(), dim=0)[0])
    blocked_widget.value = prob_blocked

    # 相機碰撞模型 + VL53L0X 雷射測距融合成單一的「被擋住」訊號。
    # 用 OR 不用加權平均:兩者盲區不同(相機容易漏掉地上的小東西,雷射遇到
    # 深色吸光表面會漏測),平均之後任一方的警告都會被稀釋,結果是照樣撞上。
    # 讀不到時 distance_m 為 None,融合邏輯單獨採信相機判斷。
    #
    # 這裡讀的是快取值,不阻塞:量測在 Arduino 上以 10Hz 進行並主動推播,
    # feedback.ArduinoBackend 的背景執行緒收下並做中位數濾波。
    distance_m = ultrasonic.distance if ultrasonic is not None else None
    distance_widget.value = -1.0 if distance_m is None else distance_m

    # ⚠️ 關鍵區分:若正前方就是鎖定中的目標,雷射讀到的近距離代表「人很近」,
    # 而不是「有障礙物」。少了這一行,機器人會把被跟隨的人當成障礙物繞開。
    range_for_blocking = None if sm.expects_target_ahead() else distance_m
    blocked = rangefinder.is_blocked(prob_blocked, range_for_blocking,
                                     distance_threshold=RANGE_BLOCK_DISTANCE)

    if blocked:
        # 官方是直接寫死 robot.left(0.3),這裡改成問狀態機。它多做三件事:
        #   1. 凍結跟丟計時,好讓避障結束後銜接回上一次的鎖定框(項目6)
        #   2. 進入 detour 狀態:轉開之後會強制橫移一段才交還跟隨,避免擺盪
        #   3. 判定繞不過去時升級成 impassable_wait,停止空轉(項目7)
        action = sm.on_blocked(now)
        handle_events(action.events)
        fb.update_lights(action.state, now)
        apply_motor(action)
        report_to_caregiver(action, prob_blocked, distance_m, now)
        state_widget.value = action.state
        image_widget.value = bgr8_to_jpeg(image)
        track_fps(now)
        return

    # --- 偵測 ---
    # SSD 走 TensorRT,不經過 autograd,所以這裡不需要 no_grad。
    detections = model(image)

    for det in detections[0]:
        bbox = det['bbox']
        cv2.rectangle(image,
                      (int(width * bbox[0]), int(height * bbox[1])),
                      (int(width * bbox[2]), int(height * bbox[3])),
                      (255, 0, 0), 2)

    matching_detections = [d for d in detections[0]
                           if d['label'] == int(label_widget.value)]

    # --- 決策全部交給狀態機(項目1、2、4、5)---
    # impassable= 參數留著給日後真的加懸崖感測器用,v1 不需要傳。
    #
    # ⚠️ 這裡傳進去的 image 已經被上面的 cv2.rectangle 畫過藍框了。
    #    藍框是純藍 (255,0,0) BGR,色相 H=120,不在桃紅或螢光黃的門檻內,
    #    所以不會污染標記偵測。改藍框顏色前要先確認這一點。
    action = sm.update(matching_detections, image, now, distance_m=distance_m)

    handle_events(action.events)
    fb.update_lights(action.state, now)
    draw_overlay(image, action, now)
    apply_motor(action)
    report_to_caregiver(action, prob_blocked, distance_m, now)

    state_widget.value = action.state
    image_widget.value = bgr8_to_jpeg(image)
    track_fps(now)


execute({'new': camera.value})

Call the block below to connect the execute function to each camera frame update.

In [ ]:
camera.unobserve_all()
camera.observe(execute, names='value')

### FPS 瓶頸分析

上面的 `FPS` 欄位告訴你「總共多快」，但不會告訴你**慢在哪一段**。
只知道「9 FPS」沒辦法決定要砍什麼。

下面這一格會停掉即時迴圈，**單獨量每一個階段**，印出分解表。跑完你會知道：

- 兩個神經網路各佔多少 → 決定先對哪一個做 TensorRT 轉檔
- JPEG 編碼（把畫面傳到瀏覽器）佔多少 → **部署時不需要畫面，這段可以整個省掉**
- 標記偵測佔多少 → 驗證「HSV 幾乎免費」這個假設成立不成立
- **理論上限 FPS** 與實際 FPS 的差距 → 差很多代表瓶頸不在模型

> ⚠️ 這一格會 `camera.unobserve_all()` 停掉即時迴圈。
> 量完想繼續跑的話，**回上面重新執行 `camera.observe(...)` 那一格**。

In [ ]:
# --- FPS 瓶頸分析:單獨量每一段 ---------------------------------------------

camera.unobserve_all()          # 先停掉即時迴圈,不然兩邊搶 GPU 量不準
robot.stop()
time.sleep(0.5)

_img = camera.value
_N = 30


def _bench(fn, n=_N):
    """回傳平均毫秒。

    兩個細節不能省:
      暖機 —— 第一次呼叫含 CUDA context 初始化與記憶體配置,會比穩態慢好幾倍
      synchronize —— CUDA 呼叫是非同步的,不同步的話只量到「把指令送出去」的時間,
                     不是「算完」的時間。少了它 GPU 的部分會看起來快得離譜
    """
    for _ in range(5):
        fn()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        fn()
    torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000.0


def _bench_collision():
    with torch.no_grad():
        collision_model(preprocess(_img))


def _bench_ssd():
    model(_img)


def _bench_marker():
    # 隨便給一個人形比例的框,只是要量時間,框準不準不影響
    verifier(_img, [0.30, 0.10, 0.70, 0.95], None)


def _bench_jpeg():
    bgr8_to_jpeg(_img)


rows = [
    ('避障模型 ResNet18(含前處理)', _bench(_bench_collision)),
    ('人物偵測 SSD(TensorRT)', _bench(_bench_ssd)),
]
if verifier is not None:
    rows.append(('標記偵測 HSV', _bench(_bench_marker)))
else:
    rows.append(('標記偵測 HSV(未啟用)', 0.0))
rows.append(('JPEG 編碼(傳畫面到瀏覽器)', _bench(_bench_jpeg)))

total = sum(ms for _n, ms in rows)

print('=' * 52)
print('每幀各階段耗時')
print('=' * 52)
for name, ms in rows:
    share = (100.0 * ms / total) if total else 0.0
    print('  %-28s %6.2f ms  %5.1f%%' % (name, ms, share))
print('-' * 52)
print('  %-28s %6.2f ms' % ('合計', total))
print('')
print('  理論上限 FPS:%.1f' % ((1000.0 / total) if total else 0.0))
print('')
print('  實際 FPS 一定比這個低 —— 還有狀態機、畫框、馬達指令,')
print('  以及 Jupyter 把畫面推到瀏覽器的網路傳輸。')
print('  兩者差距大 = 瓶頸不在模型,先想辦法關掉畫面再量一次。')
print('=' * 52)
print('')
print('⚠️ 即時迴圈已停止。要繼續跑請回上面重新執行 camera.observe(...) 那一格。')

Awesome!  If the robot is not blocked you should see boxes drawn around the detected objects in blue.  The target object (which the robot follows) will be displayed in green.

The robot should steer towards the target when it is detected.  If it is blocked by an object it will simply turn left.

You can call the code block below to manually disconnect the processing from the camera and stop the robot.

In [ ]:
import time

camera.unobserve_all()
time.sleep(1.0)
robot.stop()

if ultrasonic is not None:
    ultrasonic.close()   # 現在是 no-op:序列埠由 fb 擁有,測距在 Arduino 上
fb.close()               # 關掉三色燈、停掉背景讀取執行緒、釋放序列埠
sm.reset()
if verifier is not None:
    verifier.reset()     # 清掉標記的遮擋寬限計時

# ⚠️ 序列埠沒釋放的話,下次 fb = feedback.Feedback() 會連不上,
#    訊息是「Device or resource busy」。卡住的話在 JetBot 的 bash 執行:
#        sudo fuser -k /dev/ttyACM0

Again, let's close the camera conneciton properly so that we can use the camera in other notebooks.

In [ ]:
camera.stop()